# 03 — Testes de hipótese

Cada hipótese é testada com H0/H1, teste estatístico, tamanho de efeito, IC95%,
p-valor e **significância prática**. O veredito é consolidado ao final.

In [1]:
import sys; sys.path.insert(0, ".")
import pandas as pd
from src.analysis import hypotheses

abt = hypotheses.carregar_abt()
resultados = hypotheses.rodar_todas()
len(resultados)

6

## Tabela consolidada

In [2]:
tabela = pd.DataFrame([{
    "id": r["id"], "hipotese": r["hipotese"], "teste": r["teste"],
    "efeito": r["efeito"], "IC95": f"[{r['ic95'][0]}, {r['ic95'][1]}]",
    "p": f"{r['p']:.2e}", "n": r["n"], "veredito": r["veredito"],
} for r in resultados])
tabela[["id", "efeito", "IC95", "p", "n", "veredito"]]

,id,efeito,IC95,p,n,veredito
0,H1,-1.0942,"[-1.613, -0.5753]",3.64e-05,3824,inconclusiva (sensivel a especificacao)
1,H2,-57.6439,"[-59.6807, -55.607]",0.00e+00,4536,suportada
2,H3,-0.5300,"[-0.9717, -0.0883]",1.87e-02,4540,inconclusiva (sensivel a especificacao)
3,H4,-28.3545,"[-40.5752, -16.1339]",5.50e-06,6664,suportada
4,H5,-2.8584,"[-3.7673, -1.9494]",7.62e-10,4933,rejeitada (sinal oposto)
5,H6,-11.0048,"[-14.6222, -7.3873]",2.60e-09,6424,suportada


In [3]:
for r in resultados:
    print(f"{r['id']} — {r['hipotese']}")
    print(f"   {r['teste']}")
    print(f"   efeito={r['efeito']} ({r['unidade']})  IC95={r['ic95']}  p={r['p']:.2e}  n={r['n']}")
    print(f"   veredito: {r['veredito']}\n")

H1 — Mais medicos/1k associados a maior expectativa de vida (liquido de PIB/capita)
   PanelOLS FE pais+ano, SE cluster, controle log(PIB/cap)
   efeito=-1.0942 (anos de LE por +1 DP de medicos/1k)  IC95=[-1.613, -0.5753]  p=3.64e-05  n=3824
   veredito: inconclusiva (sensivel a especificacao)

H2 — Paises com saneamento basico > 80% tem mortalidade <5 menor
   t-test de Welch + Mann-Whitney
   efeito=-57.6439 (mortes/1k (dif. de medias))  IC95=[-59.6807, -55.607]  p=0.00e+00  n=4536
   veredito: suportada

H3 — +1 DP no gasto publico em saude (% PIB) eleva a expectativa de vida
   PanelOLS FE pais+ano, SE cluster
   efeito=-0.53 (anos de LE por +1 DP)  IC95=[-0.9717, -0.0883]  p=1.87e-02  n=4540
   veredito: inconclusiva (sensivel a especificacao)

H4 — Maior urbanizacao associada a menor mortalidade <5
   PanelOLS FE pais+ano, SE cluster
   efeito=-28.3545 (mortes/1k por +1 DP de urbanizacao)  IC95=[-40.5752, -16.1339]  p=5.50e-06  n=6664
   veredito: suportada

H5 — Apos cruzar o li

## Notas de leitura

- **H2, H4, H6 (suportadas)**: saneamento >80%, urbanização e vacinação contra
  sarampo se associam a **menor mortalidade <5**; os efeitos são grandes e robustos
  (presentes também no spec só com FE de país).
- **H1 e H3 (inconclusivas)**: o efeito de médicos e de gasto público **muda de sinal**
  entre o modelo com FE de país+ano e o modelo só com FE de país. O sinal positivo
  entre países é forte (ver EDA), mas ao absorver tendências globais o efeito within
  fica instável → não há evidência conclusiva de efeito direto.
- **H5 (rejeitada)**: no DiD 2×2, países que cruzaram o limiar UHC tiveram ganho de
  expectativa de vida **menor** que os nunca-tratados. Isso é consistente com
  **convergência** (países de renda baixa partiram de níveis menores e cresceram mais
  rápido); tratamento é endógeno e exige desenho causal mais cuidadoso (F7).

## Conclusão

O exercício separa correlação de causalidade: as hipóteses de *mortalidade infantil*
são robustas, enquanto as de *expectativa de vida* são sensíveis à especificação e o
DiD ingênuo sofre de convergência/endogeneidade — motivação direta para o componente
causal com event study e controle sintético (F7).